In [106]:
import torch 
import pandas as pd
import numpy as np 
import torch.nn as nn 
from datasets import load_dataset
from datasets import Dataset as HFDataset
from torch.utils.data import Dataset,DataLoader 
from tokenizers import Tokenizer,models,trainers,pre_tokenizers

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(device)

cpu


In [107]:
# Load Dataset from Hugginface libarary 
dataset = load_dataset('cfilt/iitb-english-hindi',split='train[:300000]')
dataset.features

{'translation': {'en': Value('string'), 'hi': Value('string')}}

In [110]:
# Delete the null values from dataset
dataset = dataset.filter(lambda x: 
                         x['translation']['en'] is not None and 
                         x['translation']['hi'] is not None and 
                         len(x['translation']['en'].strip())> 0 and
                         len(x['translation']['hi'].strip())> 0  
                        )

# Delete the duplicate values from dataset
df = dataset.flatten().to_pandas()
df = df.drop_duplicates(subset=['translation.en', 'translation.hi'],keep='first')
df = df.rename(columns={'translation.en':'en','translation.hi':'hi'})


Filter:   0%|          | 0/300000 [00:00<?, ? examples/s]

In [112]:
MAX_CHAR_LEN = 200

# Rmove the row that is more than define size 
df = df[ 
        (df["en"].str.len() <= MAX_CHAR_LEN) &
        (df['hi'].str.len() <= MAX_CHAR_LEN)
       ].reset_index(drop=True)

# Quick distribution check
print(f"Avg English length (chars): {df['en'].str.len().mean():.1f}")
print(f"Avg Hindi length (chars):   {df['hi'].str.len().mean():.1f}")
print(f"Max English length (chars): {df['en'].str.len().max()}")
print(f"Max Hindi length (chars):   {df['hi'].str.len().max()}")

Avg English length (chars): 42.2
Avg Hindi length (chars):   46.9
Max English length (chars): 200
Max Hindi length (chars):   200


In [114]:
VOCAB_SIZE = 10000

#---For english vocab---
# Train joint BPE tokenizer
en_tokenizer = Tokenizer(models.BPE(unk_token="[UNK]"))
en_tokenizer.pre_tokenizer = pre_tokenizers.Whitespace()

# Define special tokens for NMT
en_trainer = trainers.BpeTrainer(
    vocab_size = VOCAB_SIZE,
    special_tokens = ["[PAD]","[UNK]","[SOS]","[EOS]"]
)

en_tokenizer.train_from_iterator(df['en'].tolist(),trainer=en_trainer)

# ---For hindi vocab--- 
hi_tokenizer = Tokenizer(models.BPE(unk_token="[UNK]"))
hi_tokenizer.pre_tokenizer = pre_tokenizers.Whitespace()

hi_trainer = trainers.BpeTrainer(
    vocab_size=VOCAB_SIZE,
    special_tokens=["[PAD]","[UNK]","[SOS]","[EOS]"]
)

hi_tokenizer.train_from_iterator(df['hi'].tolist(),trainer=hi_trainer)

print(f"English vocab size: {en_tokenizer.get_vocab_size()}")
print(f"Hindi vocab size:   {hi_tokenizer.get_vocab_size()}")







English vocab size: 10000
Hindi vocab size:   10000


In [119]:
# Saving vocab for refetch quickliy
en_tokenizer.save("/kaggle/working/en_tokenizer.json")
hi_tokenizer.save("/kaggle/working/hi_tokenizer.json")

In [130]:
MAX_TOKEN_LEN=50

# Get the token lenth of each sentense from vocab
def get_token_len(text,tokenizer):
    return len(tokenizer.encode(text).ids)

# Removing sentense more than define size
df['en_len'] = df['en'].apply(lambda x: get_token_len(x,en_tokenizer))
df['hi_len'] = df['hi'].apply(lambda x: get_token_len(x,hi_tokenizer))

df = df[
        (df['en_len'] <= MAX_TOKEN_LEN) &
        (df['hi_len'] <= MAX_TOKEN_LEN)
        ].reset_index(drop=True)

# Quick look on final dataset
print(f"Final dataset size after token filtering: {len(df)} pairs")
print(f"Avg English tokens: {df['en_len'].mean():.1f}")
print(f"Avg Hindi tokens:   {df['hi_len'].mean():.1f}")

Final dataset size after token filtering: 152845 pairs
Avg English tokens: 9.9
Avg Hindi tokens:   12.0


In [126]:
# Splitint dataset to train,validation and test 
from sklearn.model_selection import train_test_split

train_val_df,test_df = train_test_split(df,test_size=0.1,random_state=42)
train_df,val_df = train_test_split(train_val_df,test_size=0.111, random_state=42)

train_df = train_df.reset_index(drop=True)
val_df = val_df.reset_index(drop=True)
test_df = test_df.reset_index(drop=True)

print(f"Train size:      {len(train_df)}")
print(f"Validation size: {len(val_df)}")
print(f"Test size:       {len(test_df)}")

Train size:      122290
Validation size: 15270
Test size:       15285


In [ ]:
class TranslationDataset(Dataset):
    
    def __init__(self,df,en_tokenizer,hi_tokenizer):
        self.df = df
        self.en_tokenizer = en_tokenizer
        self.hi_tokenizer = hi_tokenizer
        
        self.en_pad = en_tokenizer.token_to_ids("[PAD]")
        self.en_sos = en_tokenizer.token_to_ids("[SOS]")
        self.en_eos = en_tokenizer.token_to_ids("[EOS]")
        self.hi_pad = hi_tokenizer.token_to_ids("[PAD]")
        self.hi_sos = hi_tokenizer.token_to_ids("[SOS]")
        self.hi_eos = hi_tokenizer.token_to_ids("[EOS]")
        
    def __len__(self):
        return len(self.df)
        
    def __getitem(self,index):
        
        en_text = self.df.loc[index,'en']
        hi_text = self.df.loc[index,'hi']
        
        en_ids = self.en_tokenizer.encode(en_text).ids
        hi_ids = self.hi_tokenizer.encode(hi_text).ids

        src = [self.en_sos] + en_ids + [self.en_eos]
        trg = [self.hi_sos] + hi_ids + [self.hi_eos]

        src_tensor = torch.tensor(src,dtype= torch.long)
        trg_tensor = torch.tensor(trg,dtype= torch.long)

        return src_tensor,trg_tensor